# AI Assignment by Subhayan Das

## Import libraries

In [1]:
import pygame
import random
import time
import sys
from collections import deque
import numpy as np
import pandas as pd
import os

pygame 2.6.1 (SDL 2.28.4, Python 3.9.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Constants

In [19]:
# Constants
GRID_SIZE = 30
CELL_SIZE = 5
WINDOW_SIZE = CELL_SIZE * (2 * GRID_SIZE + 1)
BACKGROUND_COLOR = (255, 255, 255)
WALL_COLOR = (0, 0, 0)
PATH_COLOR = (255, 255, 255)
START_COLOR = (0, 255, 0)
END_COLOR = (255, 0, 0)
PATHFIND_COLOR = (0, 0, 255)
VISITED_COLOR = (255, 255, 0)
GAMMA = 0.9  # Discount factor
THETA = 1e-6  # Convergence threshold
REWARD_STEP = -1  # Penalty for each move
REWARD_GOAL = 100  # Reward for reaching the goal

## Path Finding Functions

In [20]:
def calculate_metrics(maze, start, end, screen, traversal_order, explored_nodes, path, start_time):
    # Path Complexity (Entropy) - Calculate dead ends and branching points
    def path_complexity(maze):
        dead_ends = 0
        branching_points = 0
        directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
        
        for r in range(1, len(maze) - 1, 2):
            for c in range(1, len(maze[0]) - 1, 2):
                if maze[r][c] == 0:
                    # Count dead ends (no open adjacent cells)
                    open_neighbors = 0
                    for dr, dc in directions:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]) and maze[nr][nc] == 0:
                            open_neighbors += 1
                    if open_neighbors == 0:
                        dead_ends += 1
                    elif open_neighbors > 2:
                        branching_points += 1
        return dead_ends, branching_points

    dead_ends, branching_points = path_complexity(maze)

    # Wall Density (Open Space Ratio)
    total_cells = len(maze) * len(maze[0])
    walls = sum(row.count(1) for row in maze)
    open_cells = total_cells - walls
    wall_density = walls / total_cells

    # Execution time
    execution_time = time.time() - start_time

    # Final Path Length
    final_path_length = len(path)

    # Memory Usage - Estimate based on Python object sizes
    memory_usage = sys.getsizeof(maze) + sys.getsizeof(path) + sys.getsizeof(explored_nodes)

    # Return All Metrics
    return {
        "Algorithm" : "MDP Policy Iteration",
        "Grid Size" : GRID_SIZE,
        "Path Complexity (Entropy)": (dead_ends, branching_points),
        "Wall Density (Open Space Ratio)": wall_density,
        "Number of Explored Nodes (Search Cost)": len(explored_nodes),
        "Final Path Length": final_path_length,
        "Execution Time (Seconds)": execution_time,
        "Memory Usage (Bytes)": memory_usage,
        # "Traversal Order": traversal_order
    }

In [21]:
def policy_iteration(maze):
    rows, cols = len(maze), len(maze[0])
    values = np.zeros((rows, cols))  # Initialize values
    # Initialize policy as a grid of (0, 0) direction tuples
    policy = np.empty((rows, cols), dtype=object)
    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    end = (rows - 2, cols - 2)

    # Initialize the policy grid with (0, 0) tuple
    for r in range(rows):
        for c in range(cols):
            policy[r, c] = (0, 0)  # Initialize with a default direction (right)

    # Policy Evaluation
    def policy_evaluation():
        while True:
            delta = 0
            for r in range(1, rows - 1):
                for c in range(1, cols - 1):
                    if (r, c) == end or maze[r][c] == 1:
                        continue
                    dr, dc = policy[r, c]  # This should always be a tuple
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] == 0:
                        new_value = REWARD_STEP + GAMMA * values[nr, nc]
                        delta = max(delta, abs(values[r, c] - new_value))
                        values[r, c] = new_value
            if delta < THETA:
                break

    # Policy Improvement
    def policy_improvement():
        stable = True
        for r in range(1, rows - 1):
            for c in range(1, cols - 1):
                if (r, c) == end or maze[r][c] == 1:
                    continue
                best_action = policy[r, c]
                best_value = values[r, c]
                for dr, dc in directions:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < rows and 0 <= nc < cols and maze[nr][nc] == 0:
                        new_value = REWARD_STEP + GAMMA * values[nr, nc]
                        if new_value > best_value:
                            best_value = new_value
                            best_action = (dr, dc)  # Store the best action as a tuple
                if policy[r, c] != best_action:
                    stable = False
                    policy[r, c] = best_action
        return stable

    # Run the policy iteration loop
    while True:
        policy_evaluation()
        if policy_improvement():
            break
    
    return policy


In [22]:
def mdp_traversal(maze, screen):
    policy = policy_iteration(maze)
    start = (1, 1)
    end = (len(maze) - 2, len(maze[0]) - 2)
    path = []
    traversal_order = []
    visited = set()
    r, c = start
    while (r, c) != end:
        path.append((r, c))
        traversal_order.append((r, c))
        visited.add((r, c))
        draw_maze(screen, maze, path, traversal_order)
        pygame.display.flip()
        pygame.time.delay(10)
        # Move to the next position according to the policy
        dr, dc = policy[r, c]
        nr, nc = r + dr, c + dc
        # Make sure the move is within bounds and not a wall
        if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]) and maze[nr][nc] == 0:
            r, c = nr, nc
    path.append(end)
    draw_maze(screen, maze, path, traversal_order)
    pygame.display.flip()
    return path, traversal_order, visited


In [23]:
def draw_maze(screen, maze, path, visited_cells=[]):
    for r in range(len(maze)):
        for c in range(len(maze[0])):
            color = WALL_COLOR if maze[r][c] == 1 else PATH_COLOR
            pygame.draw.rect(screen, color, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r, c in visited_cells:
        pygame.draw.rect(screen, VISITED_COLOR, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    for r, c in path:
        pygame.draw.rect(screen, PATHFIND_COLOR, (c * CELL_SIZE, r * CELL_SIZE, CELL_SIZE, CELL_SIZE))

    pygame.draw.rect(screen, START_COLOR, (CELL_SIZE, CELL_SIZE, CELL_SIZE, CELL_SIZE))
    pygame.draw.rect(screen, END_COLOR, ((len(maze[0]) - 2) * CELL_SIZE, (len(maze) - 2) * CELL_SIZE, CELL_SIZE, CELL_SIZE))


## Main Function

In [24]:
def main():
    pygame.init()

    screen_mdp = pygame.display.set_mode((WINDOW_SIZE, WINDOW_SIZE))
    pygame.display.set_caption("MDP Policy Iteration Maze Traversal")

    maze = np.load(f"./Mazes/{GRID_SIZE}.npy").tolist()

    start_time_mdp = time.time()
    mdp_path, mdp_traversal_order, mdp_visited = mdp_traversal(maze, screen_mdp)
    mdp_metrics = calculate_metrics(maze, (1, 1), (len(maze) - 2, len(maze[0]) - 2), screen_mdp, mdp_traversal_order, mdp_visited, mdp_path, start_time_mdp)

    print("MDP Policy Iteration Metrics:")
    for metric, value in mdp_metrics.items():
        print(f"{metric}: {value}")

    # Saving the metrics:
    filename = 'results.csv'
    df = pd.DataFrame([mdp_metrics])

    if os.path.exists(filename):
        existing_df = pd.read_csv(filename)
        df = pd.concat([existing_df, df], ignore_index=True)
    
    df.to_csv(filename, index=False)
    
    running = True
    while running:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
        pygame.display.flip()

    pygame.quit()

In [25]:

if __name__ == "__main__":
    main()

MDP Policy Iteration Metrics:
Algorithm: MDP Policy Iteration
Grid Size: 30
Path Complexity (Entropy): (0, 370)
Wall Density (Open Space Ratio): 0.4802472453641494
Number of Explored Nodes (Search Cost): 120
Final Path Length: 121
Execution Time (Seconds): 2.1014084815979004
Memory Usage (Bytes): 10032
